# 08 — Error structure and observed heat signal

This notebook starts two analyses that will inform the manuscript's transition to the SERRA framework:

1. **Error structure:** diagnose out-of-sample errors from the current ensemble (L), including spatial, temporal, and scale patterns.
2. **Observed heat signal:** test whether municipal IBGE/PAM yields co-vary with flowering heat exposure in the weather-yield panel, after municipality and harvest-year effects and water-stress controls.

The IBGE yield is annual and does not identify sowing dates, cultivars, or field-level management. Heat associations here are therefore observational and do not identify a causal flowering-stage effect. The heat predictor is the mean across the sowing window used by the existing hybrid model. Revisit its construction when implementing SERRA.

Run from the repository root or from `notebooks/`; outputs are written to `resultados/estrutura_erros/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT = ROOT / "resultados" / "estrutura_erros"
OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / "figuras"
FIG.mkdir(parents=True, exist_ok=True)

PRED_PATH = ROOT / "resultados" / "previsoes_fora_amostra.parquet"
BASE_PATH = ROOT / "resultados" / "base_treino_hibrido.parquet"
MUN_PATH = ROOT / "dados" / "municipios_pr.csv"


## 1. Load the out-of-fold predictions and analysis features

Each row in `previsoes_fora_amostra.parquet` is a municipality-year prediction made with that harvest year held out. The feature table contains the sowing-window average used to train the yield model.

In [ ]:
pred = pd.read_parquet(PRED_PATH)
base = pd.read_parquet(BASE_PATH)
mun = pd.read_csv(MUN_PATH)

keys = ["cod_ibge", "ano"]
assert pred.duplicated(keys).sum() == 0
assert base.duplicated(keys).sum() == 0
assert {"L_ensemble_E_SVR", "rendimento_kg_ha"}.issubset(pred.columns)
assert {"calor_floracao", "ISNA_floracao", "ISNA_enchimento"}.issubset(base.columns)

df = (pred.merge(base[keys + ["calor_floracao", "calor_reprod", "ISNA_floracao",
                               "ISNA_enchimento", "DEF_floracao", "DEF_enchimento"]],
                 on=keys, validate="one_to_one")
        .merge(mun[["cod_ibge", "municipio", "mesorregiao", "latitude", "longitude"]],
               on="cod_ibge", how="left", validate="many_to_one"))
df["pred_L_kg_ha"] = np.exp(df["L_ensemble_E_SVR"])
df["erro_log_L"] = df["log_y"] - df["L_ensemble_E_SVR"]
df["erro_kg_ha_L"] = df["rendimento_kg_ha"] - df["pred_L_kg_ha"]
df["erro_abs_kg_ha_L"] = df["erro_kg_ha_L"].abs()
print(f"{len(df):,} out-of-fold municipality-years; {df.cod_ibge.nunique()} municipalities; "
      f"{df.ano.min()}–{df.ano.max()}")
display(df[["rendimento_kg_ha", "pred_L_kg_ha", "erro_log_L", "calor_floracao"]].describe().round(2))

## 2. Error structure

Start with bias and error magnitude by harvest year and mesoregion, then check whether errors persist within municipalities across adjacent years. Spatial grouping is descriptive: municipal residuals can share unobserved regional shocks.

In [ ]:
annual_err = (df.groupby("ano").agg(n=("erro_kg_ha_L", "size"),
                                      bias_kg_ha=("erro_kg_ha_L", "mean"),
                                      mae_kg_ha=("erro_abs_kg_ha_L", "mean"),
                                      mean_obs=("rendimento_kg_ha", "mean"),
                                      mean_pred=("pred_L_kg_ha", "mean"))
              .reset_index())
regional_err = (df.groupby("mesorregiao", dropna=False).agg(n=("erro_kg_ha_L", "size"),
                                      bias_kg_ha=("erro_kg_ha_L", "mean"),
                                      mae_kg_ha=("erro_abs_kg_ha_L", "mean"),
                                      mean_obs=("rendimento_kg_ha", "mean"))
               .sort_values("mae_kg_ha", ascending=False).reset_index())

# Residual autocorrelation within municipality, lagged by harvest year.
wide = df.pivot(index="cod_ibge", columns="ano", values="erro_log_L")
lag_pairs = pd.DataFrame({"previous": wide.shift(axis=1).to_numpy().ravel(),
                          "current": wide.to_numpy().ravel()}).dropna()
r_lag, p_lag = stats.pearsonr(lag_pairs.previous, lag_pairs.current)
print(f"Adjacent-year within-municipality error correlation: r={r_lag:.3f}, p={p_lag:.3g}; "
      f"{len(lag_pairs):,} pairs")
display(annual_err.round(1))
display(regional_err.round(1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].axhline(0, color="0.4", lw=1)
axes[0].plot(annual_err.ano, annual_err.bias_kg_ha, marker="o")
axes[0].set(title="Out-of-fold bias by harvest year", xlabel="Harvest year", ylabel="Observed − predicted (kg/ha)")
axes[1].scatter(df.pred_L_kg_ha, df.erro_kg_ha_L, s=8, alpha=.2)
axes[1].axhline(0, color="0.4", lw=1)
axes[1].set(title="Error versus predicted yield", xlabel="Predicted yield (kg/ha)", ylabel="Observed − predicted (kg/ha)")
axes[2].hist(df.erro_log_L, bins=40, color="#4472C4", alpha=.85)
axes[2].axvline(0, color="0.3", lw=1)
axes[2].set(title="Log-scale out-of-fold error", xlabel="Observed − predicted log yield", ylabel="Municipality-years")
fig.tight_layout()
fig.savefig(FIG / "error_diagnostics.png", dpi=300, bbox_inches="tight")
plt.show()

annual_err.to_csv(OUT / "error_por_ano.csv", index=False)
regional_err.to_csv(OUT / "error_por_mesorregiao.csv", index=False)

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Within-year spatial structure of out-of-fold residuals, following the Season paper.
# Build a k=5 nearest-neighbor graph separately for the municipalities present each year.
rng = np.random.default_rng(20261112)
spatial_rows = []
for year, g in df.groupby("ano", sort=True):
    g = g.dropna(subset=["latitude", "longitude", "erro_log_L"]).reset_index(drop=True)
    xy = g[["latitude", "longitude"]].to_numpy()
    k = min(6, len(g))
    nn = NearestNeighbors(n_neighbors=k, metric="euclidean").fit(xy)
    neighbor_idx = nn.kneighbors(return_distance=False)[:, 1:]
    z = g["erro_log_L"].to_numpy() - g["erro_log_L"].mean()
    lag = z[neighbor_idx].mean(axis=1)
    n, k_eff = len(z), neighbor_idx.shape[1]
    I_obs = n / (k_eff * n) * (z @ lag) / (z @ z)
    I_perm = np.empty(999)
    for b in range(len(I_perm)):
        zp = rng.permutation(z)
        I_perm[b] = n / (k_eff * n) * (zp @ zp[neighbor_idx].mean(axis=1)) / (zp @ zp)
    p_perm = (1 + np.sum(np.abs(I_perm) >= abs(I_obs))) / (len(I_perm) + 1)
    spatial_rows.append({"ano": year, "n_municipios": n, "moran_I_log_error": I_obs, "p_permutation": p_perm})

spatial_by_year = pd.DataFrame(spatial_rows)
print(f"Within-year k=5 Moran's I: median={spatial_by_year.moran_I_log_error.median():.3f}; "
      f"range={spatial_by_year.moran_I_log_error.min():.3f} to {spatial_by_year.moran_I_log_error.max():.3f}; "
      f"significant years (permutation p<0.05)="
      f"{(spatial_by_year.p_permutation < .05).sum()}/{len(spatial_by_year)}")
display(spatial_by_year.round(3))
spatial_by_year.to_csv(OUT / "moran_erro_por_ano.csv", index=False)

# Descriptive variance partition: common harvest-year signal, persistent municipality bias, remainder.
yr_mean = df.groupby("ano").erro_log_L.transform("mean")
mun_mean = df.groupby("cod_ibge").erro_log_L.transform("mean")
grand = df.erro_log_L.mean()
var_total = df.erro_log_L.var(ddof=1)
var_year = np.average((df.groupby("ano").erro_log_L.mean() - grand) ** 2,
                      weights=df.groupby("ano").size())
var_municipality = np.average((df.groupby("cod_ibge").erro_log_L.mean() - grand) ** 2,
                              weights=df.groupby("cod_ibge").size())
var_within = np.mean((df.erro_log_L - yr_mean - mun_mean + grand) ** 2)
variance_partition = pd.DataFrame({
    "component": ["harvest_year_mean", "municipality_mean", "within_municipality_year_remainder"],
    "descriptive_variance": [var_year, var_municipality, var_within],
})
variance_partition["share_of_total_variance_pct"] = 100 * variance_partition.descriptive_variance / var_total
print("Descriptive (not REML) variance partition; components can overlap in an unbalanced panel:")
display(variance_partition.round(4))
variance_partition.to_csv(OUT / "error_variance_partition_descriptive.csv", index=False)

### Error persistence and interpretation

A nonzero lag correlation means some municipal error component persists across years; it can reflect stable local factors, model misspecification, or repeated measurement practices in PAM. It is not by itself evidence that the crop model should include a municipality-specific correction. Inspect the correlation alongside the year and mesoregion summaries.

## 3. Is flowering heat visible in observed yield?

Use two-way fixed-effect demeaning (municipality and year) so the association is identified by within-municipality changes relative to each year's statewide pattern. Include flowering and grain-filling water satisfaction as controls. Report a municipality-clustered standard error. This guards against treating repeated observations from the same municipality as independent.

In [ ]:
def twoway_demean(frame, columns, unit="cod_ibge", time="ano", tol=1e-10, max_iter=1000):
    """Alternating-projection residualization for unit and time fixed effects."""
    out = frame[columns].astype(float).copy()
    for _ in range(max_iter):
        old = out.to_numpy(copy=True)
        out -= out.groupby(frame[unit], sort=False).transform("mean")
        out -= out.groupby(frame[time], sort=False).transform("mean")
        if np.nanmax(np.abs(out.to_numpy() - old)) < tol:
            break
    return out

reg_cols = ["log_y", "calor_floracao", "ISNA_floracao", "ISNA_enchimento"]
reg = df.dropna(subset=reg_cols + ["cod_ibge", "ano"]).copy()
within = twoway_demean(reg, reg_cols)
y = within["log_y"].to_numpy()
X = within[["calor_floracao", "ISNA_floracao", "ISNA_enchimento"]].to_numpy()
beta = np.linalg.lstsq(X, y, rcond=None)[0]
e = y - X @ beta

# Cluster-robust sandwich covariance, clustering by municipality.
xtx_inv = np.linalg.pinv(X.T @ X)
scores = X * e[:, None]
cluster_scores = pd.DataFrame(scores, index=reg.index).groupby(reg["cod_ibge"], sort=False).sum().to_numpy()
G, N, K = len(cluster_scores), len(reg), X.shape[1]
meat = cluster_scores.T @ cluster_scores
cov = xtx_inv @ meat @ xtx_inv * (G / (G - 1)) * ((N - 1) / (N - K))
se = np.sqrt(np.diag(cov))
tval = beta / se
pval = 2 * stats.t.sf(np.abs(tval), df=G - 1)
ci = np.column_stack([beta - stats.t.ppf(.975, G-1) * se, beta + stats.t.ppf(.975, G-1) * se])
heat_result = pd.DataFrame({"term": ["heat_days_flowering", "ISNA_flowering", "ISNA_grain_filling"],
                            "estimate_log_yield": beta, "cluster_se": se, "p_value": pval,
                            "ci95_low": ci[:, 0], "ci95_high": ci[:, 1]})
display(heat_result.round(4))
print(f"N={N:,}; municipalities={G}; within R²="
      f"{1 - (e @ e) / (y @ y):.3f}")

The coefficient on `calor_floracao` is the conditional change in log yield associated with one additional flowering heat day (as represented by the existing sowing-window average). Its sign, uncertainty, and sensitivity to specification matter more than a thresholded p-value. Municipality and year fixed effects do not remove local time-varying confounders or the uncertainty in actual sowing dates.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
# Residualized bivariate display: controls and both fixed effects removed from heat and yield.
ax.scatter(within["calor_floracao"], within["log_y"], s=9, alpha=.22, color="#C0504D")
coef = np.polyfit(within["calor_floracao"], within["log_y"], 1)
xline = np.linspace(within["calor_floracao"].quantile(.01), within["calor_floracao"].quantile(.99), 100)
ax.plot(xline, np.polyval(coef, xline), color="black", lw=2)
ax.set(xlabel="Residualized flowering heat days", ylabel="Residualized log yield")
fig.tight_layout()
fig.savefig(FIG / "observed_yield_heat_association.png", dpi=300, bbox_inches="tight")
plt.show()
heat_result.to_csv(OUT / "heat_observed_fixed_effects.csv", index=False)

## 4. Sensitivity checks to complete before manuscript use

- Re-estimate without water-satisfaction controls and with a nonlinear heat term (splines or bins).
- Compare flowering heat with reproductive-period heat and phase-specific Tmax.
- Weight observations by harvested area as a sensitivity, while retaining unweighted estimates as primary because PAM municipal yields are survey estimates.
- Repeat error diagnostics for the simpler MZA-only and SERRA predictions once those out-of-fold predictions exist.
- Keep the model comparison and all feature/threshold choices inside the leave-one-year-out design.

These outputs are diagnostic starting points. Do not replace manuscript claims with them until the SERRA implementation and sensitivity checks are complete.